# Fine-Tune Laya with RLCD Strategy 1 on 60K xLAM Dataset (Cloud GPU)

**Hardware:** NVIDIA T4, V100, or A100 GPU (Google Colab / Kaggle)
**Estimated Runtime:** ~1.1 to 1.3 hours for all 60,000 examples with FP16 Automatic Mixed Precision.

In [1]:
# 1. Verify GPU availability
!nvidia-smi

Wed Oct  7 22:59:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# 2. Upload zip, extract & install dependencies
from google.colab import files

print("Click 'Choose Files' below and select 'rlcd_colab.zip' from your Desktop:")
uploaded = files.upload()

# Extract and install
!unzip -o -q rlcd_colab.zip
%cd /content/rlcd
!pip install -r requirements.txt

Click 'Choose Files' below and select 'rlcd_colab.zip' from your Desktop:


Saving rlcd_colab.zip to rlcd_colab.zip
/content/rlcd
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.3/368.3 kB 29.8 MB/s eta 0:00:00


In [8]:
# 3. Configure and run with GPU acceleration
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Read main.py
with open('main.py', 'r') as f:
    code = f.read()

# 1. Switch device from CPU to CUDA for NVIDIA GPU acceleration
code = code.replace('DEVICE = "cpu"', 'DEVICE = "cuda"')

# 2. Choose dataset size:
#    - Set to False for a fast 600-example GPU verification (~1 to 2 minutes)
#    - Set to True when you want to train on the FULL 60,000 dataset (~1 hour)
TRAIN_FULL_60K = True   # Change to True when you want to run all 60,000!

if TRAIN_FULL_60K:
    code = code.replace('DATASET_SUBSET = 600', 'DATASET_SUBSET = None')
    print("Configured to train on the FULL 60,000 dataset.")
else:
    print("Configured for 600-example GPU verification (~1-2 minutes).")

with open('main.py', 'w') as f:
    f.write(code)

print("\nStarting training on GPU:")
!python main.py

Configured to train on the FULL 60,000 dataset.

Starting training on GPU:
No HF_TOKEN found in .env; using cached local Hugging Face credentials if available.

=== Loading 'Salesforce/xlam-function-calling-60k' dataset ===
Notice: 'Salesforce/xlam-function-calling-60k' is gated on HF Hub (Dataset 'Salesforce/xlam-function-calling-60k' is a gated dataset on the Hub. You must be authenticated to access it.).
Falling back to public verified mirror 'lockon/xlam-function-calling-60k'...
Loaded Salesforce/xlam-function-calling-60k...

=== Preprocessing dataset into Laya format ===
Preparing dataset for Laya (decision_type='choice')...
Map: 100% 60000/60000 [00:05<00:00, 10810.44 examples/s]
Dataset is now Laya-input-ready.

=== Configuring RL Training Hyperparameters ===
Found 'convaiinnovations/laya' already cached at ./.cache/laya/convaiinnovations/laya/, skipping download.
Building model for 'convaiinnovations/laya'...
Loaded 'convaiinnovations/laya' for fine-tuning (device='cuda').

===

In [9]:
# 4. Display final evaluation metrics
import json
with open('laya_finetuned_xlam_600/test_evaluation_metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

{
  "baseline": {
    "accuracy": 0.8485,
    "precision": 0.8638221388961161,
    "recall": 0.8485,
    "f1": 0.8527976354639983,
    "mean_brier": -0.2504997277186109,
    "ece": 0.09978128030772011,
    "n_items": 12000
  },
  "final": {
    "accuracy": 0.991,
    "precision": 0.991008836368847,
    "recall": 0.991,
    "f1": 0.9909988946124894,
    "mean_brier": -0.01682423871234435,
    "ece": 0.008309467783818641,
    "n_items": 12000
  },
  "wall_clock_time_seconds": 3968.2131004333496
}
